# Import Libraries

In [1]:
import logging
from pathlib import Path
from huggingface_hub import snapshot_download
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
import joblib
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, classification_report, hamming_loss

# Paths
data_path = Path("../data")             # Dataset path
model_path = Path("../model")           # Model path
model_dump = model_path / "model.pkl"   # Model storage path

data_path.mkdir(parents=True, exist_ok=True)
model_path.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(
    # level = logging.INFO,       # For verbose logging
    level = logging.ERROR,        # For logging only errors
    format = "%(levelname)s: %(message)s"
)

# Fix random seed for reproducibility
np.random.seed(42)

# Initialize dictionary to store application-wide state
state_holder = dict()
logging.info("Initial Setup Process Complete")

# Fetching the dataset

In [2]:
try:
    path = snapshot_download(
        repo_id="sidzzz07/scamshield-dataset",
        repo_type="dataset",
        local_dir=str(data_path)
    )

    logging.info(f"Dataset downloaded to: {path}")

    train_df = pd.read_json(data_path / "train.jsonl", lines=True)
    test_df = pd.read_json(data_path / "test.jsonl", lines=True)
    validation_df = pd.read_json(data_path / "val.jsonl", lines=True)

except FileNotFoundError:
    logging.error("Dataset is not found, please check the dataset path and try again")
except Exception as e:
    logging.error(f"Failed to load dataset {e}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

# Training Dataset analysis

Showing random 5 rows from the training dataset

In [3]:
train_df.sample(5)

,id,text,language,source_dataset,is_scam,head1_social_engineering,head2_scam_intent,split_group,metadata
42976,real_076848,Hello sir Congratulations! Aapne ₹40 lakh ka l...,Hinglish,ysangam/Indian_Cyber_Scam_Hinglish,1,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Lottery / Prize,real_ysangam_indian_cyber_scam_hinglish,"{'has_url': False, 'has_phone': False, 'has_ot..."
36830,real_030308,ok ill be waiting,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
52659,real_025414,Cafe Iguana #01-03 Riverside point 30 merchan...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
14804,real_071896,Lol probably everyone finished their paper exc...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."
21475,real_067806,Erm u meet her help me ask eve whether she hav...,English,Smishing_Dataset,0,"{'urgency': 0, 'fear': 0, 'authority_impersona...",Legitimate / Benign,real_smishing_dataset,"{'has_url': False, 'has_phone': False, 'has_ot..."


Showing the dataset columns

In [4]:
train_df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 69286 entries, 0 to 69285
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        69286 non-null  str   
 1   text                      69286 non-null  str   
 2   language                  69286 non-null  str   
 3   source_dataset            69286 non-null  str   
 4   is_scam                   69286 non-null  int64 
 5   head1_social_engineering  69286 non-null  object
 6   head2_scam_intent         69286 non-null  str   
 7   split_group               69286 non-null  str   
 8   metadata                  69286 non-null  object
dtypes: int64(1), object(2), str(6)
memory usage: 4.8+ MB


In [5]:
language = train_df["language"].value_counts()

px.pie(
    names=language.keys(),
    values=language.values,
    title="Partition of languages in the training dataset",
)

In [6]:
scam_intent = train_df["head2_scam_intent"].value_counts()

px.bar(
    x=scam_intent.keys(),
    y=scam_intent.values,
    title="Partition of scam intent in the trianing class",
    text_auto=".0f"
)

In [7]:
is_scam = train_df["is_scam"].value_counts()

px.pie(
    names=is_scam.keys(),
    values=is_scam.values,
    title="Partition of classes",
    subtitle="(0 -> Not Spam, 1 -> Spam)",
)

## Key insights

1) Majority of the writers of the message uses English as their communication language.
2) Out of all the messages in the training dataset, most of messages are legitimate.
3) Out of all the scam messages, most of them are based on telemarketing. This reflects that fraudulent messages uses promotional offers, unsolicit marketing, fake rewards or deceptive sales tactics as a common way of scamming.
2) The dataset is imbalanced based on the partition of 36.2% of the class is spam and 63.8% of the data are not spam.

# Model Training

In [8]:
X_train = train_df["text"]
X_test = test_df["text"]

y_scam_train = train_df["is_scam"]
y_scam_test = test_df["is_scam"]

y_social_train = pd.DataFrame(
    train_df["head1_social_engineering"].tolist(),
    index=train_df.index
)
y_social_test = pd.DataFrame(
    test_df["head1_social_engineering"].tolist(),
    index=test_df.index
)

y_intent_train = train_df["head2_scam_intent"]
y_intent_test = test_df["head2_scam_intent"]

intent_encoder = LabelEncoder()
y_intent_train = intent_encoder.fit_transform(y_intent_train)
y_intent_test = intent_encoder.transform(y_intent_test)

Vectorizing text messages with TF-IDF

In [9]:
tfidf = FeatureUnion([
    (
        "word_tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            max_features=30000,
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            max_features=50000,
            min_df=2,
            sublinear_tf=True
        )
    )
])

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Scam detection Model

In [10]:
model_scam_detect = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("clf", LogisticRegression())
])

model_scam_detect.fit(X_train_tfidf, y_scam_train)

,steps,"[('smote', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
n_features_in_,int,80000
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5
Name,Type,Value


Social model used detection model

In [11]:
model_social_detect = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

model_social_detect.fit(X_train_tfidf, y_social_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](5,)","[0,1,2,3,4]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,5
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,80000
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'


Intent detection model

In [12]:
model_intent_detect = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(random_state=42))
])

model_intent_detect.fit(X_train_tfidf, y_intent_train)

,steps,"[('smote', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](12,)","[ 0, 1, 2,..., 9,10,11]"
n_features_in_,int,80000
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5
Name,Type,Value


Scam detection model prediction

In [13]:
y_scam_pred = model_scam_detect.predict(X_test_tfidf)

print("Classification Report")
print(classification_report(y_scam_test, y_scam_pred))

cm_scam = confusion_matrix(y_scam_test, y_scam_pred)

px.imshow(
    cm_scam,
    color_continuous_scale="viridis",
    text_auto=".0f",
    title="Confusion Matrix for Scam detection"
)

Classification Report
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      5191
           1       0.96      0.97      0.96      3011

    accuracy                           0.97      8202
   macro avg       0.97      0.97      0.97      8202
weighted avg       0.97      0.97      0.97      8202



Intent Detection model prediction

In [14]:
y_intent_pred = model_intent_detect.predict(X_test_tfidf)
intent_class_ids = range(len(intent_encoder.classes_))

print(
    classification_report(
        y_intent_test,
        y_intent_pred,
        labels=intent_class_ids,
        target_names=intent_encoder.classes_,
        zero_division=0
    )
)

cm_intent = confusion_matrix(y_intent_test, y_intent_pred)

px.imshow(
    cm_intent,
    color_continuous_scale="viridis",
    text_auto=".0f",
    title="Confusion Matrix for Intent detection"
)

                              precision    recall  f1-score   support

         Bank Account Freeze       1.00      0.89      0.94        18
                Courier Scam       0.90      0.97      0.93        98
   Electricity Disconnection       0.94      0.88      0.91        17
                    Fake KYC       0.97      0.97      0.97       120
General Spam / Telemarketing       0.94      0.96      0.95      2433
         Legitimate / Benign       0.98      0.97      0.98      5191
                   Loan Scam       0.89      0.96      0.93        52
             Lottery / Prize       0.96      0.93      0.95        88
      OTP / Credential Theft       0.75      0.75      0.75        12
                Telegram Job       0.83      0.83      0.83        24
           Traffic e-Challan       1.00      1.00      1.00         1
                   UPI Fraud       0.97      0.97      0.97       148

                    accuracy                           0.96      8202
                  

Social Engeineering model prediction

In [15]:
social_labels = y_social_train.columns

y_social_pred = model_social_detect.predict(X_test_tfidf)

y_social_pred = pd.DataFrame(
    y_social_pred,
    columns=social_labels,
    index=test_df.index
)

print(
    classification_report(
        y_social_test,
        y_social_pred,
        target_names=social_labels,
        zero_division=0
    )
)


print(
    "Hamming Loss:", hamming_loss(y_social_test, y_social_pred)
)

for label in social_labels:
    cm_social = confusion_matrix(y_social_test[label], y_social_pred[label], labels=[0, 1])
    fig = px.imshow(cm_social, x=["Predicted 0", "Predicted 1"], y=["Actual 0", "Actual 1"], color_continuous_scale="viridis", text_auto=".0f", title=f"Confusion Matrix - {label}")
    fig.update_layout(xaxis_title="Predicted Label", yaxis_title="Actual Label")
    fig.show()

                         precision    recall  f1-score   support

                urgency       0.87      0.98      0.92       530
                   fear       0.87      0.97      0.92       229
authority_impersonation       0.81      0.92      0.86       257
            reward_bait       0.90      0.98      0.94       985
     financial_pressure       0.79      0.94      0.86       475

              micro avg       0.86      0.97      0.91      2476
              macro avg       0.85      0.96      0.90      2476
           weighted avg       0.86      0.97      0.91      2476
            samples avg       0.21      0.21      0.21      2476

Hamming Loss: 0.011509387954157523


# Saving the model

In [16]:
model_holder = {
    "tf_idf": tfidf,
    "intent_encoder": intent_encoder,
    "model_scam_detection": model_scam_detect,
    "model_social_detect": model_social_detect,
    "model_intent_detect": model_intent_detect
}

joblib.dump(model_holder, model_dump)

['..\\model\\model.pkl']